In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

feature_cols = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
cat_features = ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember"]
y = train["Exited"].copy()

def prepare(df, feature_cols, cat_features):
    X = df[feature_cols].copy()
    for col in cat_features:
        if pd.api.types.is_float_dtype(X[col]):
            X[col] = X[col].astype(int)
    return X

X_raw = prepare(train, feature_cols, cat_features)
X_test_raw = prepare(test, feature_cols, cat_features)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X_raw, y))

## Шаг 1. Тюнинг гиперпараметров CatBoost

**Грабли по дороге**: `randomized_search` не подхватывает `cat_features` из
отдельного аргумента вызова — их нужно передать в **конструктор**
`CatBoostClassifier`, иначе падает `CatBoostError: Cannot convert 'France' to
float` (пытается интерпретировать `Geography` как число).

**Ограничили бюджет поиска**: первая версия (`iterations=1000`, сетка 4×3×4=48
комбинаций, `cv=skf` на 5 фолдах, `n_iter=15`) считалась дольше 30 минут и была
прервана. Урезали до `iterations=300`, сетки 3×2×3=18 комбинаций, `cv=3`,
`n_iter=6` — нашли параметры за разумное время, а честную оценку на 5 фолдах
делаем отдельно ниже.

In [ ]:
base_model = CatBoostClassifier(
    iterations=300, eval_metric="AUC", random_seed=42, verbose=False,
    cat_features=cat_features,
)
param_distributions = {
    "depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "l2_leaf_reg": [3, 5, 10],
}
search_results = base_model.randomized_search(
    param_distributions,
    X=X_raw, y=y,
    cv=3,
    n_iter=6,
    partition_random_seed=42,
    verbose=False,
    plot=False,
)
best_params = search_results["params"]
print("Best params:", best_params)

Best params: {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}


Найденные параметры (`depth=6, learning_rate=0.05, l2_leaf_reg=3`) **совпали** с
теми, что использовались "на глаз" в туториале 5 (кроме `l2_leaf_reg` — там был
дефолт CatBoost, тоже 3). Оцениваем честно на тех же 5 фолдах, что и раньше:

In [ ]:
def run_cv_catboost(X, y, cat_features, folds, params, label):
    oof_preds = np.zeros(len(X))
    models = []
    for fold, (train_idx, val_idx) in enumerate(folds):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = CatBoostClassifier(
            iterations=1000, eval_metric="AUC", random_seed=42, verbose=False,
            **params,
        )
        model.fit(
            X_tr, y_tr, cat_features=cat_features,
            eval_set=(X_val, y_val), early_stopping_rounds=50,
        )
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        models.append(model)
        print(f"[{label}] Fold {fold}: ROC AUC = {fold_auc:.4f}, best_iteration = {model.get_best_iteration()}")
    oof_auc = roc_auc_score(y, oof_preds)
    print(f"[{label}] OOF ROC AUC: {oof_auc:.4f}")
    return oof_auc, oof_preds, models

oof_auc_catboost, oof_catboost, models_catboost = run_cv_catboost(
    X_raw, y, cat_features, folds, best_params, "catboost tuned"
)

[catboost tuned] Fold 0: ROC AUC = 0.9272, best_iteration = 87
[catboost tuned] Fold 1: ROC AUC = 0.9292, best_iteration = 91
[catboost tuned] Fold 2: ROC AUC = 0.9189, best_iteration = 119
[catboost tuned] Fold 3: ROC AUC = 0.9409, best_iteration = 221
[catboost tuned] Fold 4: ROC AUC = 0.9288, best_iteration = 127
[catboost tuned] OOF ROC AUC: 0.9289


**Результат: 0.9289 (без изменений)** — тюнинг подтвердил, что параметры из
туториала 5 уже были близки к оптимальным для этой задачи. Реалистичный, а не
провальный результат: не всегда есть, что улучшать тюнингом.

## Шаг 2. Второй бустинг — LightGBM

In [ ]:
X_lgb = X_raw.copy()
for col in cat_features:
    X_lgb[col] = X_lgb[col].astype("category")

oof_lgb = np.zeros(len(X_lgb))
models_lgb = []
for fold, (train_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X_lgb.iloc[train_idx], X_lgb.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, random_state=42, verbosity=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    oof_lgb[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_lgb[val_idx])
    models_lgb.append(model)
    print(f"[lightgbm] Fold {fold}: ROC AUC = {fold_auc:.4f}, best_iteration = {model.best_iteration_}")

oof_auc_lgb = roc_auc_score(y, oof_lgb)
print(f"[lightgbm] OOF ROC AUC: {oof_auc_lgb:.4f}")

[lightgbm] Fold 0: ROC AUC = 0.9232, best_iteration = 81
[lightgbm] Fold 1: ROC AUC = 0.9281, best_iteration = 69
[lightgbm] Fold 2: ROC AUC = 0.9185, best_iteration = 92
[lightgbm] Fold 3: ROC AUC = 0.9411, best_iteration = 136
[lightgbm] Fold 4: ROC AUC = 0.9268, best_iteration = 73
[lightgbm] OOF ROC AUC: 0.9272


**Результат: 0.9272** — немного слабее CatBoost (0.9289), но именно такая
"немного другая, но не сильно хуже" модель полезнее всего для блендинга: она
ошибается не идентично CatBoost.

## Шаг 3. Логрегрессия на тех же 5 фолдах

In [ ]:
salary_cap = train["EstimatedSalary"].quantile(0.99)
bins = [0, 30, 40, 50, 60, 100]

train_fe = train.copy()
train_fe["EstimatedSalary"] = train_fe["EstimatedSalary"].clip(upper=salary_cap)
train_fe["AgeGroup"] = pd.cut(train_fe["Age"], bins=bins, labels=False)

fe_feature_cols = feature_cols + ["AgeGroup"]
categorical_cols_fe = ["Geography", "Gender", "NumOfProducts", "AgeGroup"]
numeric_cols_fe = [c for c in fe_feature_cols if c not in categorical_cols_fe]

X_logreg = train_fe[fe_feature_cols].copy()
oof_logreg = np.zeros(len(X_logreg))

for fold, (train_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X_logreg.iloc[train_idx], X_logreg.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols_fe),
        ("num", StandardScaler(), numeric_cols_fe),
    ])
    model = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])
    model.fit(X_tr, y_tr)
    oof_logreg[val_idx] = model.predict_proba(X_val)[:, 1]

oof_auc_logreg = roc_auc_score(y, oof_logreg)
print(f"[logreg] OOF ROC AUC (5-fold, comparable): {oof_auc_logreg:.4f}")

[logreg] OOF ROC AUC (5-fold, comparable): 0.9213


0.9213 на 5-fold CV, чуть ниже, чем 0.9246 из туториала 4 (там был один
`train_test_split`, плюс чуть другой набор признаков) — напоминание, что одна
оценка на одном сплите может отличаться от усреднённой по 5 фолдам.

## Шаг 4. Блендинг

In [ ]:
best_alpha_2, best_auc_2 = 0, 0
for alpha in np.arange(0, 1.01, 0.05):
    blend = alpha * oof_catboost + (1 - alpha) * oof_lgb
    auc = roc_auc_score(y, blend)
    if auc > best_auc_2:
        best_alpha_2, best_auc_2 = alpha, auc
print(f"CatBoost+LightGBM: best alpha={best_alpha_2:.2f}, blended OOF={best_auc_2:.4f}")

best_weights_3, best_auc_3 = None, 0
step = 0.1
for w_cb in np.arange(0, 1.01, step):
    for w_lgb in np.arange(0, 1.01 - w_cb, step):
        w_lr = max(1 - w_cb - w_lgb, 0)
        blend = w_cb * oof_catboost + w_lgb * oof_lgb + w_lr * oof_logreg
        auc = roc_auc_score(y, blend)
        if auc > best_auc_3:
            best_auc_3 = auc
            best_weights_3 = (round(w_cb, 2), round(w_lgb, 2), round(w_lr, 2))
print(f"CatBoost+LightGBM+LogReg: best weights (cb, lgb, lr)={best_weights_3}, blended OOF={best_auc_3:.4f}")

CatBoost+LightGBM: best alpha=0.75, blended OOF=0.9291
CatBoost+LightGBM+LogReg: best weights (cb, lgb, lr)=(0.7, 0.2, 0.1), blended OOF=0.9291


**Результат: 0.9289 → 0.9291.** Небольшой, но положительный прирост (+0.0002) —
ровно тот скромный эффект, который и предсказывался в плане туториала. LogReg с
весом 0.1 добавил чуть-чуть поверх смеси CatBoost+LightGBM (0.7/0.2 → 0.7/0.2/0.1),
хотя сама по себе она заметно слабее (0.9213).

## Шаг 5. Финальный сабмит

In [ ]:
# CatBoost на 100% train
final_iters_cb = int(np.mean([m.get_best_iteration() for m in models_catboost]))
final_catboost = CatBoostClassifier(
    iterations=final_iters_cb, eval_metric="AUC", random_seed=42, verbose=False, **best_params,
)
final_catboost.fit(X_raw, y, cat_features=cat_features)
test_proba_cb = final_catboost.predict_proba(X_test_raw)[:, 1]

# LightGBM на 100% train
final_iters_lgb = int(np.mean([m.best_iteration_ for m in models_lgb]))
X_test_lgb = X_test_raw.copy()
for col in cat_features:
    X_test_lgb[col] = X_test_lgb[col].astype("category")
final_lgb = lgb.LGBMClassifier(
    n_estimators=final_iters_lgb, learning_rate=0.05, random_state=42, verbosity=-1,
)
final_lgb.fit(X_lgb, y)
test_proba_lgb = final_lgb.predict_proba(X_test_lgb)[:, 1]

# LogReg на 100% train
test_fe = test.copy()
test_fe["EstimatedSalary"] = test_fe["EstimatedSalary"].clip(upper=salary_cap)
test_fe["AgeGroup"] = pd.cut(test_fe["Age"], bins=bins, labels=False)
X_test_logreg = test_fe[fe_feature_cols].copy()

preprocessor_final = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols_fe),
    ("num", StandardScaler(), numeric_cols_fe),
])
final_logreg = Pipeline([
    ("preprocess", preprocessor_final),
    ("classifier", LogisticRegression(max_iter=1000)),
])
final_logreg.fit(X_logreg, y)
test_proba_lr = final_logreg.predict_proba(X_test_logreg)[:, 1]

w_cb, w_lgb, w_lr = best_weights_3
final_proba = w_cb * test_proba_cb + w_lgb * test_proba_lgb + w_lr * test_proba_lr

submission = pd.DataFrame({"id": test["id"], "Exited": final_proba})
submission.to_csv("../submissions/ensemble.csv", index=False)

## Итог туториала 6

```
Модель: блендинг CatBoost(0.7) + LightGBM(0.2) + LogReg(0.1)
OOF ROC AUC: 0.9291 (было 0.9289 у одного CatBoost)
Kaggle public score:  0.93957 (было 0.93935)
Kaggle private score: 0.93703 (было 0.93687)
```

**Результат — примерно 3-е место** финального лидерборда (между текущими
#2 — 0.93715 и #3 — 0.93688).

Главные выводы этапа:
- **Тюнинг не дал прироста** — найденные `randomized_search` параметры совпали
  с дефолтными из туториала 5. Не всегда есть что тюнить; это нормальный,
  информативный результат, а не ошибка процесса.
- **Блендинг дал скромный, но реальный прирост** (+0.0002 OOF, +0.00016 на
  приватном лидерборде) — LightGBM и особенно слабая логрегрессия чуть-чуть
  дополнили CatBoost за счёт разных ошибок, а не за счёт собственной силы.
- Поймали и исправили реальный баг CatBoost (`cat_features` в конструктор, а
  не в `randomized_search`) и урезали неудачно большой бюджет поиска (30+ минут
  без результата) до практичного — тоже часть работы с бустингом на практике.